In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [12]:
from datasets import load_dataset
import tokenizers
from torch.utils.data import random_split, DataLoader

**Positional Encodings:**
- A positional encoding is a dense vector that encodes the position of a token within a sentence: the ith positional encoding is added to the token embedding of the ith token in each sentence. Can be simply implemented using an embedding layer: Just add embedding #0 to the representation of token #0, add embedding #1 to the representation of token 1.

In [3]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X):
        return self.dropout(X + self.pos_embed[:X.size(1)])

# the inputs have shape: (batch_size, seq_length, embedding_size), but we are adding positional encodings of shape: [sequence_length, embedding_size]. this wprks because of broadcasting rules, the ith positional embedding is added to the ith token representation of each sentence in the batch

**Also, might be worth checking out other positional encoding techniques and 'Relative Positional Encoding'**

**Multi-Head Attention:**
- The multi-head attention (MHA) layer is based on scaled dot-product attention, a variant of dot-product attention that scales down the similarity scores by a constant factor.
- The MHA layer splits the values, keys, and queries across multiple heads: this way, each head can focus on specific characteristics of the token. The first linear layer lets the model choose which xtics each head should focus on. 
- Final representation layer lets the model reorganize the representation as it pleases
- Idea is that the linear layers at the start sort of prep the attention for different xtics to look out for in a token. Then we split into multiple heads of attention and each head is solely focused on learning a different xtic type. we concat the output and use the final linear layer that ensures the model concats this representation the way it wants

**Masks:**
- attn_mask:
  - Boolean mask of shape (Lq, Lk) that we use to control which key tokens each query should ignore (True to ignore, False to attend)
- key_padding_mask
  - Boolean mask of shape (B, Lk) to locate the padding tokens in each key

In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1,2)

    def forward(self, query, key, value, attn_mask = None, key_padding_mask = None):
        q = self.split_heads(self.q_proj(query)) # (B, h, Lq, dq)
        k = self.split_heads(self.k_proj(key)) # (B, h, Lk, dk)
        v = self.split_heads(self.v_proj(value)) # (B, h, Lv, dv), with Lk = Lv
        # compute the raw dot product scores
        scores = q @ k.transpose(2,3) / self.d**0.5 # (B, h, Lq, Lk)

        # Masking support
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf)
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2) # (B, 1, 1, Lk)
            scores = scores.masked_fill(mask, -torch.inf)
            
        weights = scores.softmax(dim=-1) # (B, h, Lq, Lk)
        # apply dropout
        Z = self.dropout(weights) @ v # (B, h, Lq, dv)
        Z = Z.transpose(1,2) # (B, Lq, h, dv)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d) # (B, Lq, h*d)
        return (self.out_proj(Z), weights) # (B, Lq, h x d)

**Building the rest of the transformer:**
- The rest of the architecture: more straightforward. Starting with the encoder block. The following implementation closely matches the encoder block except it sprinkles a bit of dropout after the self attention layer and after both dense layers in the feedforward module

In [5]:
# the transformer encoder block:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        attn, _ = self.self_attn(src, src, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(
            self.linear2(
                self.dropout(
                    self.linear1(Z).relu()
                )
            )
        )
        return self.norm2(Z + ff)

Note:
- the default feedforward block is composed of a first linear layer that expands the dimensionality to 2048 (by default), followed by a nonlinearity (RELU) then a second linear layer that projects the data back down to the original embedding size (the model dimension, d_model). This reverse bottleneck (d_model to dim_feedforward to d_model) increases the expressive power of the nonlinearity, allowing the model to learn much richer combination of features. (an idea explored further in the mobilenetv2 paper)
- In the encoder, the src_mask argument is generally not used, since the encoder allows each token to atend to all tokens, even the ones located after it. However, the key_padding_mask ought to be set appropriately
- The decoder block:

In [6]:
# the decoder block:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.multihead_attn = MultiheadAttention(d_model, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask = None, memory_mask = None, tgt_key_padding_mask = None, memory_key_padding_mask = None):
        attn1, _ = self.self_attn(tgt, tgt, tgt, attn_mask = tgt_mask, key_padding_mask = tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))

        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask = memory_mask, key_padding_mask = memory_key_padding_mask)

        Z = self.norm2(Z + self.dropout(attn2))

        ff = self.dropout(
            self.linear2(
                self.dropout(
                    self.linear1(Z).relu()
                )
            )
        )
        return self.norm3(Z + ff)

Extra modules:

In [7]:
from copy import deepcopy

In [42]:
class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm = None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(encoder_layer) for _ in range(num_layers)])
        self.norm = norm

    def forward(self, src, mask = None, src_key_padding_mask = None):
        Z = src 
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [43]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm = None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(decoder_layer) for _ in range(num_layers)])
        self.norm = norm

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask = None, memory_key_padding_mask=None):
        Z = tgt
        for layer in self.layers:
            Z = layer(Z, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [44]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm1 = nn.LayerNorm(d_model)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, norm1)
        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm2 = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, norm2)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        memory = self.encoder(src, src_mask, src_key_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
        return output

**Building an English-to-Spanish Transformer:**
- we now build our NMT Transformer module: we use our positional embedding module and pytorch's nn.Transformer. Pytorch's nn.Transformer because it is faster than the above custom built transformer module

**Preparing the dataset:**

In [13]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=['validation', 'test']
)

In [14]:
nmt_original_valid_set

Dataset({
    features: ['source_text', 'target_text', 'source_lang', 'target_lang'],
    num_rows: 197299
})

In [15]:
nmt_original_valid_set[0]

{'source_text': '100 euros for the whole day.',
 'target_text': '100 euros por toda la jornada.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [18]:
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [20]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [21]:
def train_eng_spa(): # a generator function to iterate over all training text
    for pair in nmt_train_set:
        yield pair['source_text']
        yield pair['target_text']

max_length = 500
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [22]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device), self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [23]:
def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 64
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size, collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size, collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size, collate_fn=nmt_collate_fn)

In [48]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)
        # self.transformer = nn.Transformer(
        #     embed_dim, num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers, batch_first=True
        # )
        self.transformer = Transformer(
            embed_dim, num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers,dropout=dropout
        )
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, pair):
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] * 2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        out_decoder = self.transformer(src_embeds, tgt_embeds, src_key_padding_mask=src_pad_mask, memory_key_padding_mask=src_pad_mask, tgt_mask=causal_mask, tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0, 2, 1)

In [26]:
torch.full((4, 4),True)

tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]])

In [28]:
torch.triu(torch.full((4,4), True), diagonal=1)

tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

alternatively instead of using the above technqiues for causal masking, we could have called:

In [29]:
nn.Transformer.generate_square_subsequent_mask(5)

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

we then call the transformer, passing it it the source and target embeddings, as well as all the appropriate masks

In [31]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

In [49]:
torch.manual_seed(42)

nmt_tr_model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0, num_heads=4, num_layers=2, dropout=0.1).to(device)

# if device == 'mps':
# on mps devices, using a custom transformer, because the nn.Transformer module explodes during training n mps
# nmt_tr_model = Transformer(
#     d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2
# ).to(device)

n_epochs = 20
xentropy = nn.CrossEntropyLoss(ignore_index=0) # ignore padding tokens
optimizer = torch.optim.NAdam(nmt_tr_model.parameters())

val_loss = [0] * n_epochs
train_loss = [0] * n_epochs

for epoch in range(n_epochs):
    nmt_tr_model.train()
    # loop thru the training batch
    for pair, labels in nmt_train_loader:
        pair, labels = pair.to(device), labels.to(device)
        # zero grad
        optimizer.zero_grad()
        out = nmt_tr_model(pair)
        loss = xentropy(out, labels)
        # back prop
        loss.backward()
        # step:
        optimizer.step()
        train_loss[epoch] += loss.item() * labels.size(0)
    train_loss[epoch] /= len(nmt_train_loader.dataset)

    # validation
    nmt_tr_model.eval()
    for pair, labels in nmt_valid_loader:
        pair, labels = pair.to(device), labels.to(device)
        out = nmt_tr_model(pair)
        loss = xentropy(out, labels)
        val_loss[epoch] += loss.item() * labels.size(0)
    val_loss[epoch] /= len(nmt_valid_loader.dataset)

    print(f"Epoch: {epoch+1} | Train Loss: {train_loss[epoch]} | Val Loss: {val_loss[epoch]}")

Epoch: 1 | Train Loss: 4.136635313907801 | Val Loss: 3.258064720440926
Epoch: 2 | Train Loss: 3.023963832878789 | Val Loss: 2.5857884703586382
Epoch: 3 | Train Loss: 2.597803186971977 | Val Loss: 2.336388086511348
Epoch: 4 | Train Loss: 2.385591189854304 | Val Loss: 2.191229279127087
Epoch: 5 | Train Loss: 2.247888804951034 | Val Loss: 2.0685093433072708
Epoch: 6 | Train Loss: 2.1484786314743727 | Val Loss: 2.015116297334689
Epoch: 7 | Train Loss: 2.0730887481706626 | Val Loss: 1.9380250154747554
Epoch: 8 | Train Loss: 2.0116513520083634 | Val Loss: 1.912530777361329
Epoch: 9 | Train Loss: 1.957915388113922 | Val Loss: 1.8716773414998507
Epoch: 10 | Train Loss: 1.9151909544400572 | Val Loss: 1.8410236251577772
Epoch: 11 | Train Loss: 1.874913601958867 | Val Loss: 1.8206716751605196
Epoch: 12 | Train Loss: 1.8416578653955535 | Val Loss: 1.8151750197098815
Epoch: 13 | Train Loss: 1.8150470688594413 | Val Loss: 1.7906133436999585
Epoch: 14 | Train Loss: 1.7857929350462933 | Val Loss: 1.76

In [51]:
torch.save(nmt_tr_model.state_dict(), "my_nmt_tr_model.pt")

In [52]:
# a translate function
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [ ]:
nmt_tr_model.eval()
translate(nmt_tr_model, "I like to play soccer with my friends at the beach")

' Me gusta jugar fútbol con mis amigos en la playa . </s>'

Trained for 20 epochs and works really well. - Imagine a much bigger one trained on a much larger dataset

In [54]:
# free up GPU RAM
del nmt_tokenizer_model, nmt_test_loader, nmt_train_loader, nmt_valid_loader
del nmt_tr_model, nmt_train_set, nmt_valid_set, optimizer
del xentropy